In [ ]:
!pip -q install torch-geometric
import os
import glob
import pickle
import random
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (
    NNConv,
    GlobalAttention
)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
GRAPH_DIR = "/content/drive/MyDrive/ds003844_processed/adaptive_graph_batches"
SAVE_DIR = "/content/drive/MyDrive/ds003844_processed/graph_autoencoder"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
graph_files = sorted(glob.glob(os.path.join(GRAPH_DIR, "*.pkl")))
all_graphs = []
for file in tqdm(graph_files):
    with open(file, "rb") as f:
        graphs = pickle.load(f)
    all_graphs.extend(graphs)
print("Total graphs:", len(all_graphs))

In [ ]:
pyg_graphs = []
for A in tqdm(all_graphs):
    A = A.astype(np.float32)
    n = A.shape[0]
    x = torch.ones((n,1),dtype=torch.float)
    rows, cols = np.nonzero(A)
    edge_index = torch.tensor(
        np.vstack([rows, cols]),
        dtype=torch.long
    )
    edge_attr = torch.tensor(
        A[rows, cols],
        dtype=torch.float
    ).view(-1,1)
    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        num_nodes=n
    )
    pyg_graphs.append(data)
print("Total PyG graphs:",len(pyg_graphs))

In [ ]:
random.seed(42)
random.shuffle(pyg_graphs)
split = int(0.9*len(pyg_graphs))
train_graphs = pyg_graphs[:split]
val_graphs = pyg_graphs[split:]
print(len(train_graphs))
print(len(val_graphs))

In [ ]:
train_loader = DataLoader(
    train_graphs,
    batch_size=32,
    shuffle=True
)
val_loader = DataLoader(
    val_graphs,
    batch_size=32,
    shuffle=False
)
print("Ready.")

In [ ]:
from torch_geometric.nn import NNConv
edge_network1 = nn.Sequential(
    nn.Linear(1, 64),
    nn.ReLU(),
    nn.Linear(64, 64)
)
edge_network2 = nn.Sequential(
    nn.Linear(1, 128),
    nn.ReLU(),
    nn.Linear(128, 64 * 128)
)
edge_network3 = nn.Sequential(
    nn.Linear(1, 128),
    nn.ReLU(),
    nn.Linear(128, 128 * 128)
)

In [ ]:
from torch_geometric.nn import NNConv, AttentionalAggregation
class GraphEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = NNConv(
            in_channels=1,
            out_channels=64,
            nn=edge_network1,
            aggr="mean"
        )
        self.conv2 = NNConv(
            in_channels=64,
            out_channels=128,
            nn=edge_network2,
            aggr="mean"
        )
        self.conv3 = NNConv(
            in_channels=128,
            out_channels=128,
            nn=edge_network3,
            aggr="mean"
        )
        self.pool = AttentionalAggregation(
            gate_nn=nn.Sequential(
                nn.Linear(128,64),
                nn.ReLU(),
                nn.Linear(64,1)
            )
        )
    def forward(self,data):
        x = data.x
        edge_index = data.edge_index
        edge_attr = data.edge_attr
        batch = data.batch
        x = self.conv1(x, edge_index, edge_attr)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_attr)
        x = F.relu(x)
        x = self.conv3(x, edge_index, edge_attr)
        x = F.relu(x)
        node_embeddings = x
        graph_embedding = self.pool(node_embeddings, batch)
        return node_embeddings, graph_embedding

In [ ]:
class PairwiseDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.Linear(384,256),
            nn.ReLU(),
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,1)
        )
    def forward(self,
                graph_embedding,
                node_embeddings,
                edge_index,
                batch):
        src = edge_index[0]
        dst = edge_index[1]
        src_embed = node_embeddings[src]
        dst_embed = node_embeddings[dst]
        graph_embed = graph_embedding[batch[src]]
        decoder_input = torch.cat(
            [
                graph_embed,
                src_embed,
                dst_embed
            ],
            dim=1
        )
        pred = self.decoder(decoder_input)
        return pred

In [ ]:
class GraphAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = GraphEncoder()
        self.decoder = PairwiseDecoder()
    def forward(self, data):
        node_embeddings, graph_embedding = self.encoder(data)
        reconstructed_edges = self.decoder(
            graph_embedding,
            node_embeddings,
            data.edge_index,
            data.batch
        )
        return reconstructed_edges, node_embeddings, graph_embedding

In [ ]:
model = GraphAutoencoder().to(device)
print(model)

In [ ]:
import torch.nn.functional as F
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=10
)
epochs = 150

In [ ]:
def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for data in tqdm(loader):
        data = data.to(device)
        optimizer.zero_grad()
        reconstructed_edges, _, _ = model(data)
        target_edges = data.edge_attr
        loss = F.mse_loss(
            reconstructed_edges,
            target_edges
        )
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
@torch.no_grad()
def validate(model, loader):
    model.eval()
    total_loss = 0
    for data in loader:
        data = data.to(device)
        reconstructed_edges, _, _ = model(data)
        loss = F.mse_loss(
            reconstructed_edges,
            data.edge_attr
        )
        total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
batch = next(iter(train_loader)).to(device)
reconstructed_edges, node_embeddings, graph_embeddings = model(batch)
print("Predicted:", reconstructed_edges.shape)
print("Target:", batch.edge_attr.shape)
print("Node Embeddings:", node_embeddings.shape)
print("Graph Embeddings:", graph_embeddings.shape)

In [ ]:
best_val_loss = float("inf")
for epoch in range(1, epochs + 1):
    train_loss = train_epoch(
        model,
        train_loader,
        optimizer
    )
    val_loss = validate(
        model,
        val_loader
    )
    scheduler.step(val_loss)
    print(
        f"Epoch {epoch:03d} | "
        f"Train {train_loss:.6f} | "
        f"Val {val_loss:.6f}"
    )
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            model.encoder.state_dict(),
            SAVE_DIR + "/graph_encoder.pt"
        )
        print("Best model saved.")

In [ ]:
torch.save(model.state_dict(),
           "/content/drive/MyDrive/ds003844_processed/graph_autoencoder_best.pt")

In [ ]:
torch.save(model.encoder.state_dict(),
           "/content/drive/MyDrive/ds003844_processed/graph_encoder.pt")

In [ ]:
encoder.eval()
all_embeddings = []
with torch.no_grad():
    for data in pyg_graphs:
        data = data.to(device)
        z = encoder(data)      # <-- pass the whole Data object
        all_embeddings.append(z.cpu())
all_embeddings = torch.cat(all_embeddings, dim=0)
torch.save(
    all_embeddings,
    "/content/drive/MyDrive/ds003844_processed/spatial_graph_embeddings.pt"
)
print(all_embeddings.shape)